# Notebook 04 — Original SmolLM2 vs the trained LoRA adapter

Evaluate the **selected full-run adapter** on the exact 300 three-shot questions saved by Notebook 02c. Reuse the original answers as the baseline. There are 300 new adapter predictions and no new training.

**Kaggle setup:** enable Internet, select a T4 GPU, and attach these three **saved outputs** using Add Input:

1. **Notebook 01:** original data bundle.
2. **Notebook 02c:** completed pilot, including `selection.json`, `predictions.jsonl`, and manifests.
3. **Notebook 03:** completed training output, including `full/adapter` and its manifests.

Folders and standalone ZIPs are supported. Attach the saved output files rather than only the `.ipynb` notebook. Do not attach Notebook 02b in place of 02c or select the smoke adapter.

The report keeps the official scorer fixed and shows accuracy, answer parseability, improved/regressed answers, and uncertainty. The supplementary arithmetic audit is explicitly exploratory: some original correct equations lacked `Final answer:` and were scored invalid. Apply the same audit to both models. This remains a development comparison on previously inspected test sets.


## 1. Fetch the shared code from GitHub

The exact commit is printed and saved. The original evaluator files are checked against the baseline's saved fingerprints before inference, so changes to stopping or extraction cannot silently alter the comparison.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)

Repository: https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git
Code commit: f2cbfd25f98bd425532d615f7733ce1f1c2e3644
Project checkout: /tmp/math-project-mp3t651c


From https://github.com/sahan24854/Fine-Tuning-LLM-for-math
 * branch            main       -> FETCH_HEAD
 * [new branch]      main       -> origin/main


## 2. Discover and verify all three saved inputs before loading model weights

The code checks artifact checksums; original data/model revision; saved question IDs, prompts, and demonstrations; baseline scores; full-training scope; and selected adapter weights. It recomputes the question plan only to verify the saved selection, never to substitute a new selection.

A saved full-training folder is selected automatically, even when its sibling smoke folder is present. If more than one distinct baseline or full-training run is attached, select the intended path with `MATH_BASELINE_PATH` or `MATH_TRAINING_PATH` before rerunning this cell. `MATH_BUNDLE_PATH` selects a particular original data bundle if necessary.


In [2]:
import hashlib
import shutil

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings before continuing.")
_ = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
INPUT_ROOT = Path("/kaggle/input")
BUNDLE_ROOT = Path(os.environ.get("MATH_BUNDLE_PATH", str(INPUT_ROOT)))
BASELINE_ROOT = Path(os.environ.get("MATH_BASELINE_PATH", str(INPUT_ROOT)))
TRAINING_ROOT = Path(os.environ.get("MATH_TRAINING_PATH", str(INPUT_ROOT)))
BUNDLE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_bundle.py"), str(BUNDLE_ROOT)], text=True).strip())
BASELINE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_results.py"), str(BASELINE_ROOT), "--kind", "pilot"], text=True).strip())
TRAINING = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_results.py"), str(TRAINING_ROOT), "--kind", "training"], text=True).strip())
CONFIG_PATH = PROJECT / "configs/comparison_v1.json"
CONFIG_HASH = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
verification = subprocess.check_output([sys.executable, str(PROJECT / "scripts/compare_pilot.py"),
    "--bundle", str(BUNDLE), "--baseline", str(BASELINE), "--training", str(TRAINING),
    "--config", str(CONFIG_PATH), "--output", "/kaggle/working/comparison-verification-only", "--verify-only"],
    text=True, cwd=PROJECT)
VERIFIED = json.loads(verification)
OUTPUT = Path("/kaggle/working") / f"comparison-v1-{VERIFIED['baseline_sha256'][:8]}-{VERIFIED['training_sha256'][:8]}-{CONFIG_HASH[:8]}-{CODE_COMMIT[:8]}"
print("Data bundle:", BUNDLE)
print("Original pilot:", BASELINE)
print("Full training:", TRAINING)
print("Verified inputs:", json.dumps(VERIFIED, indent=2))
print("Output:", OUTPUT)


Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04
Data bundle: /kaggle/input/notebooks/sahan24854/01-data-and-protocol/data-protocol-20261019T082909Z-1cb02106
Original pilot: /kaggle/input/notebooks/sahan24854/02c-smollm2-three-shot-baseline-pilot/pilot-v2-b9d3419aa714-d7b6ded7-80439bf1
Full training: /kaggle/input/notebooks/sahan24854/03-teach-smollm2-maths-with-lora/lora-v1-b9d3419aa714-1100d832-25f43858/full
Verified inputs: {
  "inputs_verified": true,
  "predictions": 300,
  "comparison_contract_sha256": "b34b624c4aa58b70efe3c71d011075b8cc8f6f815631556e1553abe8f4050051",
  "baseline_sha256": "1274df15d41d457065d887aed3c7bc8f8b642e5c31f9198035bba38625ea024a",
  "training_sha256": "c1a577f42d943230dc125ddd894fda6f6dd686505fe64196fe068935b95b3063",
  "selected_checkpoint": "checkpoint-633",
  "base_model": {
    "id": "HuggingFaceTB/SmolLM2-360M",
    "revision": "f8027fd0eaeea54caa13c31d31b9fdc459c38b49",
    "revision_policy": "Resolve once in notebook 01; every subse

## 3. Prepare the pinned inference and adapter environment

Use the same core PyTorch/Transformers/CUDA settings as the original pilot, plus the PEFT package needed to load the adapter. The new complete package fingerprint is recorded separately. One T4 is used; batch size four, greedy generation, eager FP16 base inference, and the 512-token cap remain fixed.

The script refuses differences in recorded core framework, hardware, precision, EOS/padding, and stopping settings. The adapter is frozen, unmerged, and loaded over the pinned base revision. The original bundle tokenizer is used.

The existing nonfatal `wrapt` startup warning and `torch_dtype` deprecation message may appear. The CUDA check and contract tests must still pass.


In [3]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-comparison-env-"))
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/setup_environment.py"),
                    "--environment", str(ENVIRONMENT), "--profile", "training"], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
INFERENCE_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
probe = """
import torch, peft
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable: check Kaggle GPU settings.')
x = torch.ones((32, 32), device='cuda', dtype=torch.float16)
assert (x @ x).sum().item() == 32768
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0), 'PEFT:', peft.__version__)
"""
_ = subprocess.run([str(PYTHON), "-I", "-c", probe], check=True, env=INFERENCE_ENV)


Creating isolated environment with Python 3.13.15


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Processing /tmp/math-pip-bootstrap-jdhj455q/pip-25.3-py3-none-any.whl


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 31.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 73.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 110.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 19.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 29.3 MB/s  0:00:00



...

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


No broken requirements found.
Environment ready: /tmp/math-comparison-env-kmk98thj/bin/python


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


PyTorch: 2.7.1+cu118 CUDA: 11.8 GPU: Tesla T4 PEFT: 0.17.1


## 4. Check comparison contracts on CPU

Tests use tiny local models/adapters and synthetic outputs. They check selected-checkpoint verification, frozen adapter loading, paired improvements/regressions, bootstrap behavior, the narrow arithmetic rule, ZIP/symlink discovery, input tampering, and interruption recovery. These tests are separate from the 300 scored SmolLM2 predictions.


In [4]:
_ = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", "test_comparison.py", "-b", "-v"],
                    cwd=PROJECT, check=True, env=INFERENCE_ENV)


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
test_input_checks_detect_tampering_and_wrong_bundle (test_comparison.AdapterComparisonTests.test_input_checks_detect_tampering_and_wrong_bundle) ... ok
test_partial_resume_reports_matched_questions_and_preserves_complete_results (test_comparison.AdapterComparisonTests.test_partial_resume_reports_matched_questions_and_preserves_complete_results) ... ok
test_real_adapter_cli_and_verification_only_path (test_comparison.AdapterComparisonTests.test_real_adapter_cli_and_verification_only_path) ... ok
test_discovery_supports_zip_symlinks_and_refuses_smoke_or_ambiguous_inputs (test_comparison.ComparisonContractTests.test_discovery_supports_zip_symlinks_and_refuses_smoke_or_ambiguous_inputs) ... ok
test_equation_audit_accepts_only_complete_matching_single_equation (test_comparison.ComparisonContractTests.test_equation_audit_accepts_only_complete_matching_single_equation) ... ok
test_paired_coun

## 5. Generate 300 adapter answers and compare with saved baseline answers

The exact original job order, prompts, demonstrations, per-row stopping, and official numeric scorer are reused. Invalid answers remain incorrect in the denominator.

Results are flushed after each batch. Rerun this cell in the same session to resume after interruption; changed code/config/runtime/input identities require a new run directory. An interrupted batch is regenerated in its original composition without appending duplicates. Original input artifacts remain read-only.

During a partial run, each comparison uses only questions completed by both models. Final paired bootstrap intervals use 10,000 resamples per dataset (seed 42); arithmetic resamples within the original number-range/operation strata. These intervals do not turn previously inspected tests into an untouched holdout.


In [5]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/compare_pilot.py"),
    "--bundle", str(BUNDLE), "--baseline", str(BASELINE), "--training", str(TRAINING),
    "--config", str(CONFIG_PATH), "--output", str(OUTPUT)],
    check=True, cwd=PROJECT, env=INFERENCE_ENV)
SUMMARY = json.loads((OUTPUT / "summary.json").read_text())
if not SUMMARY["execution_complete"]:
    raise RuntimeError("Comparison is incomplete; resume this cell with the same settings.")
print("Completed adapter predictions:", SUMMARY["predictions"])
for dataset, group in SUMMARY["groups"].items():
    print(dataset, json.dumps(group, indent=2))
print("Supplementary arithmetic:", json.dumps(SUMMARY["supplementary_arithmetic"], indent=2))


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Loading HuggingFaceTB/SmolLM2-360M at f8027fd0eaeea54caa13c31d31b9fdc459c38b49 on Tesla T4


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded in 7.0s
Saved-baseline paired comparison: 0/300 adapter predictions saved
Saved 4/300 adapter predictions (gsm8k_test)
Saved 8/300 adapter predictions (gsm8k_test)
Saved 12/300 adapter predictions (gsm8k_test)
Saved 16/300 adapter predictions (gsm8k_test)
Saved 20/300 adapter predictions (gsm8k_test)
Saved 24/300 adapter predictions (gsm8k_test)
Saved 28/300 adapter predictions (gsm8k_test)
Saved 32/300 adapter predictions (gsm8k_test)
Saved 36/300 adapter predictions (gsm8k_test)
Saved 40/300 adapter predictions (gsm8k_test)
Saved 44/300 adapter predictions (gsm8k_test)
Saved 48/300 adapter predictions (gsm8k_test)
Saved 52/300 adapter predictions (gsm8k_test)
Saved 56/300 adapter predictions (gsm8k_test)
Saved 60/300 adapter predictions (gsm8k_test)
Saved 64/300 adapter predictions (gsm8k_test)
Saved 68/300 adapter predictions (gsm8k_test)
Saved 72/300 adapter predictions (gsm8k_test)
Saved 76/300 adapter predictions (gsm8k_test)
Saved 80/300 adapter predictions (gsm8k_t

## 6. Review paired answers and retain the comparison ZIP

The report shows original/tuned accuracy, answer parseability, paired changes and confidence intervals, cap hits, and expandable raw answers. Check both improvements and regressions. A correct final number does not prove every reasoning step is valid.

The supplementary arithmetic rule additionally recognizes a target containing only one complete equation matching the question's operands/operator, such as `18 + 9 = 27`. It rejects cap-limited targets, extra prose, alternative equations, and later-question answers. The rule was designed after inspecting the original baseline and is exploratory. It does not replace the official score or fully isolate mathematical reasoning from formatting.

Keep the original data, pilot, and training ZIPs as well as this comparison ZIP. Share `summary.json` and `paired_predictions.jsonl`, or save the executed notebook to GitHub for review. Do not select another checkpoint or change scoring based on these test outcomes.


In [6]:
from IPython.display import HTML, display
RESULT = json.loads((OUTPUT / "result_manifest.json").read_text())
for name, expected in RESULT["files_sha256"].items():
    if hashlib.sha256((OUTPUT / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"Comparison result checksum mismatch: {name}")
display(HTML((OUTPUT / "report.html").read_text()))
ARCHIVE = Path(shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT))
print("Comparison ZIP:", ARCHIVE)
print("Save Version with outputs. Retain original Notebook 01, 02c, and 03 outputs.")


Dataset,Paired,Original accuracy,Tuned accuracy,Change (percentage points),95% paired interval (pp),Improved / regressed,Original / tuned parse rate
gsm8k_test,100/100,5.0%,8.0%,+3.0,-4.0 to +10.0,8 / 5,75.0% / 95.0%
svamp_test,100/100,20.0%,12.0%,-8.0,-16.0 to +0.0,5 / 13,84.0% / 98.0%
arithmetic_test,100/100,4.0%,54.0%,+50.0,+44.0 to +56.0,51 / 1,23.0% / 97.0%


Comparison ZIP: /kaggle/working/comparison-v1-1274df15-c1a577f4-9f1d3995-f2cbfd25.zip
Save Version with outputs. Retain original Notebook 01, 02c, and 03 outputs.
